# Program 5: Deep Learning for Text Document Classification

**Course:** Deep Learning Lab — 7th Semester, CSE (AI & ML)  
**Environment:** Google Colab (Runtime → Change runtime type → GPU recommended for CNN / transfer learning)

This notebook follows Experiment 5 in the uploaded *DL Lab Manual*. Run cells from top to bottom. The program saves generated artifacts in `/content/DL_Lab_Outputs/Program_5`.

**Note:** Training accuracy, test accuracy, loss values and predictions can vary between runs and hardware. The notebook generates the actual output files when executed in Colab.

In [ ]:
# Import the package or class needed in this experiment.
import numpy as np
# Import the package or class needed in this experiment.
import pandas as pd
# Import the package or class needed in this experiment.
import tensorflow as tf
# Import the package or class needed in this experiment.
from pathlib import Path
# Import the package or class needed in this experiment.
from tensorflow.keras import Sequential
# Import the package or class needed in this experiment.
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense

# Perform this processing step in the experiment.
tf.keras.utils.set_random_seed(42)
# Create or define the folder used to store program outputs.
OUT = Path("/content/DL_Lab_Outputs/Program_5")
# Create or define the folder used to store program outputs.
OUT.mkdir(parents=True, exist_ok=True)

# IMDB movie review dataset; labels: 0=negative, 1=positive
# Assign the resulting value to `(x_train, y_train), (x_test, y_test)`.
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.imdb.load_data(num_words=10000)
# Assign the resulting value to `x_train`.
x_train = tf.keras.preprocessing.sequence.pad_sequences(x_train, maxlen=200, padding="post", truncating="post")
# Assign the resulting value to `x_test`.
x_test = tf.keras.preprocessing.sequence.pad_sequences(x_test, maxlen=200, padding="post", truncating="post")

# Assign the resulting value to `model`.
model = Sequential([
    # Assign the resulting value to `Embedding(10000, 32, input_length`.
    Embedding(10000, 32, input_length=200),
    # Perform this processing step in the experiment.
    GlobalAveragePooling1D(),
    # Assign the resulting value to `Dense(64, activation`.
    Dense(64, activation="relu"),
    # Assign the resulting value to `Dense(1, activation`.
    Dense(1, activation="sigmoid")
# Perform this processing step in the experiment.
])
# Configure the optimizer, loss function, and evaluation metrics.
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
# Train the model using the prepared training data.
history = model.fit(x_train, y_train, epochs=5, batch_size=32,
                    # Assign the resulting value to `validation_split`.
                    validation_split=0.2, verbose=1)
# Evaluate the trained model on the test dataset.
loss, accuracy = model.evaluate(x_test, y_test, verbose=0)
# Generate predictions for the supplied input samples.
probabilities = model.predict(x_test[:5], verbose=0).ravel()
# Perform this processing step in the experiment.
predictions = (probabilities >= 0.5).astype(int)

# Display the current result or evaluation metric.
print(f"Test Accuracy: {accuracy:.4f}")
# Control execution using this condition, loop, or context.
for i, (label, prob) in enumerate(zip(predictions, probabilities), start=1):
    # Display the current result or evaluation metric.
    print(f"Review {i}: {'Positive' if label else 'Negative'} | probability={prob:.4f}")

# Save experiment results to an output file.
pd.DataFrame({"actual": y_test[:5], "predicted": predictions, "positive_probability": probabilities}).to_csv(OUT/"review_predictions.csv", index=False)
# Save experiment results to an output file.
pd.DataFrame(history.history).to_csv(OUT/"training_history.csv", index=False)
# Save the trained model or reusable objects.
model.save(OUT/"imdb_text_classifier.keras")